# From DN to reflectance: Landsat 8 over Brno

[![Open in Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OpenSkyLab-space/notebooks/blob/main/dn_to_reflectance_brno_en.ipynb)

**OpenSkyLab · notebook for the lab** [From DN to reflectance](https://openskylab.space/edu_viz_dn_to_reflectance.html) · [česká verze](dn_to_reflectance_brno_cs.ipynb)

The lab walks through the chain from digital number to radiance to top-of-atmosphere (TOA) and surface (BOA) reflectance and shows what dark object subtraction (DOS) and the empirical line correct. Here you compute the whole chain on two real Landsat 8 images of Brno and compare the result with Sentinel-2 L2A reflectance from the same week.

**Goal:** convert DN to radiance and TOA reflectance, show why DN from two days are not comparable, measure the atmospheric effect over water, and compare DOS and the empirical line with surface reflectance.

**Data:** Landsat 8 OLI Collection 1 Level-1 from the Google Cloud public archive: `LC08_L1TP_190026_20200808_20200821_01_T1` (8 Aug 2020, sun elevation 54°) and `LC08_L1TP_190026_20190227_20190309_01_T1` (27 Feb 2019, sun elevation 30°), bands B1–B7, BQA and MTL. For comparison Sentinel-2 L2A `S2B_33UXQ_20200808_0_L2A` (8 Aug 2020) from Earth Search. A 12 × 12 km window from the Brno reservoir to central Brno, about 25 MB.

**Tasks:** five. Write your answers in the *Your answer* cells below the tasks. Lines marked `# ◀ parameter` are meant to be changed; run the cell again with Shift + Enter.

**Time:** about 45 min. You need a Google account (Colab); nothing is installed on your computer. Run the cells in order from top to bottom.

## 1. Setup and data

> **💻 What the code does:** Installs the rasterio library, which reads raster data (GeoTIFF) directly from the internet. The other libraries are already in Colab.

In [ ]:
%pip install -q rasterio

> **💻 What the code does:** For a summer and a winter Landsat 8 scene it downloads the metadata (MTL) and reads only a square window from all seven optical bands and the quality mask. From the mask it selects pixels without clouds and shadows and prints the sun elevation, the Earth–Sun distance and the share of clear pixels.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import requests
from rasterio.warp import Resampling, reproject

GCS = "https://storage.googleapis.com/gcp-public-data-landsat/LC08/01/190/026/"
SCENES = {
    "summer": "LC08_L1TP_190026_20200808_20200821_01_T1",
    "winter": "LC08_L1TP_190026_20190227_20190309_01_T1",
}
# Lower-left corner of the window in UTM 33N (m) and its side.
X0, Y0, SIZE = 607620, 5446530, 12000
BANDS = {
    1: "coastal 443",
    2: "blue 482",
    3: "green 561",
    4: "red 655",
    5: "NIR 865",
    6: "SWIR1 1609",
    7: "SWIR2 2201",
}


def mtl(scene):
    txt = requests.get(f"{GCS}{scene}/{scene}_MTL.txt", timeout=60).text
    return {
        k.strip(): v.strip().strip('"')
        for k, v in (line.split(" = ") for line in txt.splitlines() if " = " in line)
    }


def read(scene, band):
    with rasterio.Env(
        GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR",
        GDAL_HTTP_MAX_RETRY=5,
        GDAL_HTTP_RETRY_DELAY=2,
    ):
        with rasterio.open(f"/vsicurl/{GCS}{scene}/{scene}_{band}.TIF") as r:
            w = (
                rasterio.windows.from_bounds(X0, Y0, X0 + SIZE, Y0 + SIZE, r.transform)
                .round_offsets()
                .round_lengths()
            )
            read.transform = r.window_transform(w)
            return r.read(1, window=w).astype(float)


L8 = {}
for season, scene in SCENES.items():
    meta = mtl(scene)
    dn = {b: read(scene, f"B{b}") for b in BANDS}
    L8_transform = read.transform
    qa = read(scene, "BQA").astype(int)
    # No clouds and no high-confidence shadows.
    ok = ((qa >> 4) & 1 == 0) & ((qa >> 7) & 3 < 3) & (dn[1] > 0)
    L8[season] = {"meta": meta, "dn": dn, "ok": ok, "transform": L8_transform}
    print(
        f"{season}: {scene}, sun elevation {float(meta['SUN_ELEVATION']):.1f}°, Earth–Sun distance {meta['EARTH_SUN_DISTANCE']} AU, clear pixels {ok.mean():.1%}"
    )

## 2. From DN to radiance and TOA reflectance

Radiance: L = M_L · DN + A_L. TOA reflectance: ρ = (M_ρ · DN + A_ρ) / sin(sun elevation). The coefficients are in the MTL file; M_ρ already includes the Earth–Sun distance and the solar irradiance of the band.

> **💻 What the code does:** Using the MTL coefficients it converts the digital numbers of each band to radiance and to top-of-atmosphere (TOA) reflectance corrected for sun elevation. It shows a table of median DN, radiance and reflectance in summer and winter and their ratios.

In [ ]:
for s in L8.values():
    m, dn = s["meta"], s["dn"]
    sun = np.sin(np.radians(float(m["SUN_ELEVATION"])))
    s["rad"] = {
        b: float(m[f"RADIANCE_MULT_BAND_{b}"]) * dn[b]
        + float(m[f"RADIANCE_ADD_BAND_{b}"])
        for b in BANDS
    }
    s["toa"] = {
        b: (
            float(m[f"REFLECTANCE_MULT_BAND_{b}"]) * dn[b]
            + float(m[f"REFLECTANCE_ADD_BAND_{b}"])
        )
        / sun
        for b in BANDS
    }

rows = []
for b, name in BANDS.items():
    r = {"band": f"B{b} {name}"}
    for season, s in L8.items():
        ok = s["ok"]
        r[f"DN {season}"] = np.median(s["dn"][b][ok])
        r[f"radiance {season}"] = np.median(s["rad"][b][ok])
        r[f"ρ TOA {season}"] = np.median(s["toa"][b][ok])
    rows.append(r)
tab = pd.DataFrame(rows).set_index("band")
tab["ratio DN summer/winter"] = tab["DN summer"] / tab["DN winter"]
tab["ratio ρ summer/winter"] = tab["ρ TOA summer"] / tab["ρ TOA winter"]
tab.round(3)

> **✏️ Task 1.** How much do the median DN differ between summer and winter, and how much the median TOA reflectance? How much of the DN difference is explained by sun elevation alone (compare the ratio of the sines)? Why does even TOA reflectance not agree completely between the dates (state of vegetation, possible snow, atmosphere)?

*Your answer:* 

## 3. Surface reflectance from Sentinel-2 as a reference

Sentinel-2 L2A from 8 Aug 2020 (the same day) has BOA reflectance from the Sen2Cor model. We resample it to the Landsat grid (average into 30 m pixels) and pair bands of similar wavelength. Do not expect a perfect match: it is a different sensor with slightly different bands, an image taken about 20 minutes later and from a different viewing angle.

> **💻 What the code does:** Downloads a Sentinel-2 L2A scene with surface (BOA) reflectance from the Element 84 catalogue and resamples its seven bands and the SCL class map to the Landsat grid (averaging into 30 m pixels). It selects pixels where both datasets are valid and plots the Landsat TOA and Sentinel-2 BOA colour images side by side with the same colour stretch.

In [ ]:
item = requests.get(
    "https://earth-search.aws.element84.com/v1/collections/sentinel-2-l2a/items/S2B_33UXQ_20200808_0_L2A",
    timeout=60,
).json()
PAIR = {
    1: "coastal",
    2: "blue",
    3: "green",
    4: "red",
    5: "nir08",
    6: "swir16",
    7: "swir22",
}
shape = L8["summer"]["dn"][1].shape
grid = L8["summer"]["transform"]


def s2(key, res=Resampling.average):
    out = np.zeros(shape, np.float32)
    with rasterio.Env(
        GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR",
        GDAL_HTTP_MAX_RETRY=5,
        GDAL_HTTP_RETRY_DELAY=2,
    ):
        with rasterio.open("/vsicurl/" + item["assets"][key]["href"]) as r:
            reproject(
                rasterio.band(r, 1),
                out,
                dst_transform=grid,
                dst_crs="EPSG:32633",
                resampling=res,
            )
    return out


# Since processing baseline 04.00 DN carry a +1000 offset unless the catalogue has
# removed it.
p = item["properties"]
off = (
    -1000
    if float(p.get("s2:processing_baseline", "0")) >= 4
    and not p.get("earthsearch:boa_offset_applied", False)
    else 0
)
boa = {b: (s2(k) + off) / 10000 for b, k in PAIR.items()}
scl = s2("scl", Resampling.mode)
summer = L8["summer"]
valid = summer["ok"] & np.isin(scl, [4, 5, 6]) & (boa[1] > 0)
print(f"pixels for comparison: {valid.sum()}")
fig, ax = plt.subplots(1, 2, figsize=(11, 5))


def rgb(d):
    return np.clip(np.dstack([d[4], d[3], d[2]]) / 0.25, 0, 1)


ax[0].imshow(rgb(summer["toa"]))
ax[0].set_title("Landsat 8, TOA reflectance, 8 Aug 2020")
ax[1].imshow(rgb(boa))
ax[1].set_title("Sentinel-2 L2A, BOA reflectance, 8 Aug 2020")
for a in ax:
    a.axis("off")
plt.show()

> **✏️ Task 2.** Compare the two images with the same colour stretch. Why does the TOA reflectance image look brighter, bluer and less contrasting? Which surfaces change most between TOA and BOA, dark or bright ones? Explain it with the additive contribution of the atmosphere.

*Your answer:* 

## 4. Blue water above the atmosphere

The Brno reservoir and the Svratka are almost black in the NIR. We compare their TOA and BOA reflectance in all bands: the difference is mainly light scattered by the atmosphere (path radiance).

> **💻 What the code does:** Finds water from the Sentinel-2 NDWI index and the water class in SCL. For each band it shows the median TOA and BOA reflectance over water and their difference, i.e. the contribution of the atmosphere.

In [ ]:
ndwi = (boa[3] - boa[5]) / (boa[3] + boa[5])
water = valid & (ndwi > 0.3) & (scl == 6)
w = pd.DataFrame(
    {
        "ρ TOA (Landsat)": [np.median(summer["toa"][b][water]) for b in BANDS],
        "ρ BOA (Sentinel-2)": [np.median(boa[b][water]) for b in BANDS],
    },
    index=[f"B{b} {n}" for b, n in BANDS.items()],
)
w["TOA − BOA"] = w.iloc[:, 0] - w.iloc[:, 1]
print(f"water pixels: {water.sum()}")
w.round(4)

> **✏️ Task 3.** How does TOA − BOA over water change with wavelength? Compare the ratio between bands B1 and B4 with Rayleigh's law (scattering ∝ λ⁻⁴). Why is water blue above the atmosphere although it is dark in all bands at the surface?

*Your answer:* 

## 5. Dark object subtraction (DOS)

We assume that the darkest pixels in the scene have almost zero surface reflectance (1 % is the usual choice, Chavez 1996). Whatever they have on top is path radiance, which we subtract from the whole scene. DOS corrects only the additive part of the atmosphere.

> **💻 What the code does:** Dark object subtraction: in each band it takes the TOA reflectance at percentile `pct` as the dark object and subtracts its difference from the assumed reflectance `rho_dark` from the whole scene. It shows the error (RMSE) of the original and corrected data against Sentinel-2 and the mean deviation (bias).

In [ ]:
pct = 0.1  # ◀ parameter: percentile taken as the dark object (%)
rho_dark = 0.01  # ◀ parameter: assumed true reflectance of the dark object
dos = {
    b: summer["toa"][b]
    - (np.percentile(summer["toa"][b][summer["ok"]], pct) - rho_dark)
    for b in BANDS
}


def rmse(a, b, m=valid):
    return float(np.sqrt(np.mean((a[m] - b[m]) ** 2)))


def bias(a, b, m=valid):
    return float(np.mean(a[m] - b[m]))


res = pd.DataFrame(
    {
        "dark object ρ TOA": [
            np.percentile(summer["toa"][b][summer["ok"]], pct) for b in BANDS
        ],
        "RMSE TOA": [rmse(summer["toa"][b], boa[b]) for b in BANDS],
        "RMSE DOS": [rmse(dos[b], boa[b]) for b in BANDS],
        "bias DOS": [bias(dos[b], boa[b]) for b in BANDS],
    },
    index=[f"B{b}" for b in BANDS],
)
res.round(4)

> **✏️ Task 4.** In which bands does DOS reduce the error against BOA and in which does it make it larger? Why is the DOS bias negative in all bands (what does DOS do with the multiplicative part, the atmospheric transmittance)? Set `pct = 2` and then `rho_dark = 0`: what happens? When does DOS fail completely (a winter scene with snow and no dark objects, haze over only part of the scene)?

*Your answer:* 

## 6. Empirical line

The empirical line corrects reflectance with a slope and an offset. In the field the
reference can be physical calibration panels with known reflectance.
In this exercise we use **reference pixels**: their reference reflectance
is taken from a Sentinel-2 L2A image.

For each band we look for a line between the Landsat TOA reflectance and the reference
BOA reflectance. We then correct the whole band with this line:

$$
\widehat{\rho}_{\mathrm{BOA}} = a\,\rho_{\mathrm{TOA}} + b.
$$

- $\rho_{\mathrm{TOA}}$ is the input top-of-atmosphere reflectance.
- $\widehat{\rho}_{\mathrm{BOA}}$ is our estimate of the surface reflectance.
- $a$ is the **slope** (`slope`): it scales the values.
- $b$ is the **offset** (`offset`): it adds the same correction to every value.

Reflectance, slope and offset are dimensionless. For a simplified model of the atmosphere

$$
\rho_{\mathrm{TOA}} \approx \rho_{\mathrm{path}} + T\,\rho_{\mathrm{BOA}}
\quad\Rightarrow\quad
a \approx \frac{1}{T},\qquad b \approx -\frac{\rho_{\mathrm{path}}}{T},
$$

where $T$ is the atmospheric transmittance and $\rho_{\mathrm{path}}$ the contribution of
atmospheric scattering expressed as reflectance. This physical reading is approximate:
here we compare two different sensors, and the reference BOA is itself a product of processing.

### 6.1 Select reference pixels and correct the image

In each band we select, according to Sentinel-2, the `n_reference` darkest and the same
number of brightest valid pixels. With `n_reference = 300` the line is therefore fitted
to **600 pairs of values**, not to two group means. Different bands
may select different pixels.

Indexing with `[valid]` picks the values at the valid locations shared by both images,
in the same order. `np.argsort` returns their indices from darkest to
brightest. `np.polyfit(..., deg=1)` computes the slope and offset of the line
from the selected pairs. We then apply the line to the whole Landsat band.

**After changing `n_reference`, run all three cells of this part again (6.1–6.3).**

In [ ]:
# Number of dark and of bright reference pixels; twice as many in total.
n_reference = 300  # ◀ parameter: pixels in each group
empirical = {}
coefficients = {}

for band in BANDS:
    toa_image = summer["toa"][band]
    toa_valid = toa_image[valid]
    boa_valid = boa[band][valid]

    sorted_indices = np.argsort(boa_valid)
    dark_indices = sorted_indices[:n_reference]
    bright_indices = sorted_indices[-n_reference:]
    reference_indices = np.concatenate((dark_indices, bright_indices))

    toa_reference = toa_valid[reference_indices]
    boa_reference = boa_valid[reference_indices]
    slope, offset = np.polyfit(toa_reference, boa_reference, deg=1)

    empirical[band] = slope * toa_image + offset
    coefficients[band] = (slope, offset)

### 6.2 Compare the errors of all three approaches

The table shows the slope and offset in each band and the RMSE of the original TOA reflectance,
of the DOS correction and of the empirical line against the Sentinel-2 BOA reference:

$$
\mathrm{RMSE} = \sqrt{\frac{1}{N}\sum_{i=1}^{N}
\left(\rho_i^{\mathrm{estimate}}-\rho_i^{\mathrm{reference}}\right)^2}.
$$

$N$ is the number of all valid pixels and $i$ denotes one of them. A smaller RMSE
means better agreement with the reference. All methods are evaluated on the same pixels,
including those used to fit the line, so this is not an independent accuracy assessment.

In [ ]:
rows = []
for band in BANDS:
    slope, offset = coefficients[band]
    rows.append(
        {
            "band": f"B{band}",
            "slope a": slope,
            "offset b": offset,
            "RMSE TOA": rmse(summer["toa"][band], boa[band]),
            "RMSE DOS": rmse(dos[band], boa[band]),
            "RMSE emp. line": rmse(empirical[band], boa[band]),
        }
    )

comparison = pd.DataFrame(rows).set_index("band")
print(comparison.round(4))

### 6.3 Show the agreement with Sentinel-2

The horizontal axis shows the reference BOA, the vertical axis the reflectance from the given approach.
The black 1 : 1 line marks perfect agreement. Colours distinguish the blue band B2,
the red B4 and the near-infrared B5.

For clarity we plot only **4,000 randomly selected valid pixels**,
the same for all plots and bands. The fixed seed `0` makes the selection
repeatable. This sample is used only for plotting; it changes neither the calibration nor the RMSE.

In [ ]:
rng = np.random.default_rng(0)
n_valid = np.count_nonzero(valid)
plot_indices = rng.choice(n_valid, size=4000, replace=False)

methods = [("TOA", summer["toa"]), ("DOS", dos), ("empirical line", empirical)]
band_colors = {2: "tab:blue", 4: "tab:red", 5: "tab:purple"}
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for axis, (name, estimates) in zip(axes, methods):
    for band, color in band_colors.items():
        reference_values = boa[band][valid]
        estimated_values = estimates[band][valid]
        axis.scatter(
            reference_values[plot_indices],
            estimated_values[plot_indices],
            s=2,
            alpha=0.4,
            c=color,
            label=f"B{band}",
        )
    axis.axline((0, 0), slope=1, color="k", lw=0.8)
    axis.set(
        xlim=(0, 0.5), ylim=(0, 0.5), xlabel="ρ BOA Sentinel-2", ylabel=name, title=name
    )

axes[0].legend()
plt.tight_layout()
plt.show()

> **✏️ Task 5.** Why does the empirical line have a negative offset and a slope different from 1 in the blue bands? What do they express physically? Which method has the smallest RMSE in each band, and why can DOS not correct the slope? Set `n_reference = 20` and run cells 6.1–6.3 again: how do the slope, offset and RMSE change? Why are two calibration panels with different known reflectance enough to determine the line in the field, but must they be in the same image and under the same illumination?

*Your answer:* 

> **✏️ Extension (optional).** Compare your results with a classmate who ran the notebook for another place: did you get a similar TOA − BOA difference over water and a similar empirical line slope? What could make them differ (another day, other haze, another sun elevation)?

*Your answer:* 

---
### Data sources and references

- Landsat 8 OLI Collection 1 Level-1, U.S. Geological Survey; Google Cloud public archive (`gcp-public-data-landsat`). USGS data have no use restrictions, attribution requested.
- USGS (2019). *Landsat 8 (L8) Data Users Handbook*, version 5.0, ch. 5 (DN to radiance and TOA reflectance).
- Copernicus Sentinel-2 data 2020, ESA (L2A, Sen2Cor), AWS Open Data registry (`sentinel-cogs`), Earth Search catalogue (Element 84). Free use with attribution.
- Chavez, P. S. (1988). An improved dark-object subtraction technique for atmospheric scattering correction of multispectral data. *Remote Sensing of Environment* 24, 459–479. https://doi.org/10.1016/0034-4257(88)90019-3
- Chavez, P. S. (1996). Image-based atmospheric corrections – revisited and improved. *Photogrammetric Engineering and Remote Sensing* 62(9), 1025–1036.
- Smith, G. M., Milton, E. J. (1999). The use of the empirical line method to calibrate remotely sensed data to reflectance. *International Journal of Remote Sensing* 20, 2653–2662. https://doi.org/10.1080/014311699211994

### How to cite

OpenSkyLab (2026). From DN to reflectance: Landsat 8 over Brno [notebook, teaching resource], version 1.0. https://github.com/OpenSkyLab-space/notebooks/blob/main/dn_to_reflectance_brno_en.ipynb. Licence CC BY 4.0.

Notebook text and code: OpenSkyLab, licence [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Data are under their providers' licences (see above).

*This resource was developed by the OpenSkyLab team with the assistance of the generative AI Claude (Anthropic), used for drafting code and text. The authors are responsible for the content.*